# cDVGAN Evaluation Notebook

Compare the **TensorFlow cDVGAN (epoch 210)** and **PyTorch cDVGAN** generators using:

1. Waveform visualisation — real vs TF vs PyTorch
2. UMAP dimensionality reduction
3. Gravity Spy classification (requires `gengli` / `gravityspy`)

**Before running:**
- Set `TF_GENERATOR_PATH` to your local `.keras` file.
- Copy PyTorch checkpoints from CIT into `weights/pytorch/` (see the README there).
- Set `PYTORCH_EPOCH` to the epoch you want to evaluate.

## Configuration

In [ ]:
import os
import sys
from pathlib import Path

# ── Path configuration ────────────────────────────────────────────────────────
PROJECT_ROOT = Path(".").resolve()          # run from project root
DATA_DIR     = PROJECT_ROOT / "data"        # .npy files (gitignored, see scripts/setup_local.sh)
PLOTS_DIR    = PROJECT_ROOT / "evaluation_plots"
os.makedirs(PLOTS_DIR, exist_ok=True)

# TF generator
TF_GENERATOR_PATH = PROJECT_ROOT / "weights" / "tensorflow" / "generator_210.keras"

# PyTorch generator: set to None to skip PyTorch evaluation
PYTORCH_WEIGHTS_DIR = PROJECT_ROOT / "weights" / "pytorch"
PYTORCH_EPOCH = 500          # e.g. 25, 50, ..., 500

# Generation parameters
NOISE_DIM        = 100
NUM_CLASSES      = 7
SAMPLES_PER_CLASS = 5000     # samples to generate per class for UMAP / GravitySpy
LABEL_ORDER = [
    "Blip", "Fast_Scattering", "Koi_Fish",
    "Low_Frequency_Burst", "Scattered_Light", "Tomte", "Whistle"
]

print("Project root :", PROJECT_ROOT)
print("TF weights   :", TF_GENERATOR_PATH)
print("PT weights   :", PYTORCH_WEIGHTS_DIR / f"generator_{PYTORCH_EPOCH}.pt")

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import umap.umap_ as umap

try:
    import scienceplots
    plt.style.use(["science", "no-latex"])
except ImportError:
    pass  # scienceplots optional

matplotlib.rcParams.update({"text.usetex": False})

## 1. Load Real Data

In [ ]:
X_real = np.load(DATA_DIR / "glitch_GAN_samples_scaled_balanced.npy")
y_onehot = np.load(DATA_DIR / "glitch_GAN_labels_balanced.npy")
label_order_npy = np.load(DATA_DIR / "glitch_GAN_label_order.npy")

# Use the canonical order from configuration
y_real = np.array(LABEL_ORDER)[np.argmax(y_onehot, axis=1)]

print(f"Real signals : {X_real.shape}")
print(f"Labels       : {y_onehot.shape}")
print(f"Classes      : {LABEL_ORDER}")

In [ ]:
# ── Plot one example per class ────────────────────────────────────────────────
def plot_per_class_row(X, y, label_order, color, save_name=None):
    fig, axes = plt.subplots(1, len(label_order), figsize=(3 * len(label_order), 3))
    for i, lbl in enumerate(label_order):
        idx = np.where(y == lbl)[0]
        signal = X[np.random.choice(idx)]
        axes[i].plot(signal, lw=0.8, color=color)
        axes[i].set_xticks([])
        axes[i].set_yticks([])
        axes[i].set_title(lbl.replace("_", " "), fontsize=13)
    plt.tight_layout()
    if save_name:
        plt.savefig(PLOTS_DIR / f"{save_name}.pdf", bbox_inches="tight")
    plt.show()

plot_per_class_row(X_real, y_real, LABEL_ORDER, color="C0", save_name="real_waveforms_row")

## 2. TensorFlow Generator (epoch 210)

Loads `generator_210.keras` — the TF cDVGAN checkpoint that produced good results
at epoch 210 during the original training run.

In [ ]:
try:
    import keras
    tf_generator = keras.models.load_model(str(TF_GENERATOR_PATH), compile=False)
    print(f"Loaded TF generator from: {TF_GENERATOR_PATH}")
    tf_generator.summary(line_length=80)
    TF_AVAILABLE = True
except Exception as e:
    print(f"[WARNING] Could not load TF generator: {e}")
    TF_AVAILABLE = False

In [ ]:
if TF_AVAILABLE:
    import tensorflow as tf

    num_examples = NUM_CLASSES * SAMPLES_PER_CLASS
    noise_tf = tf.random.normal(shape=(num_examples, NOISE_DIM))
    class_vectors_tf = np.repeat(np.eye(NUM_CLASSES), SAMPLES_PER_CLASS, axis=0)

    X_fake_tf = tf_generator.predict([noise_tf, class_vectors_tf], batch_size=256)
    y_fake_tf  = np.repeat(LABEL_ORDER, SAMPLES_PER_CLASS)

    print(f"TF generated  : {X_fake_tf.shape}  ({SAMPLES_PER_CLASS} per class)")
else:
    X_fake_tf = None
    y_fake_tf  = None

In [ ]:
if TF_AVAILABLE:
    plot_per_class_row(X_fake_tf, y_fake_tf, LABEL_ORDER,
                       color="C1", save_name="tf_generated_waveforms_row")

## 3. PyTorch Generator

Loads `generator_{PYTORCH_EPOCH}.pt` from `weights/pytorch/`.
Copy checkpoint files from CIT before running this section
(see `weights/pytorch/README.md`).

In [ ]:
import torch

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from cdvgan.model_components import Generator

pt_ckpt_path = PYTORCH_WEIGHTS_DIR / f"generator_{PYTORCH_EPOCH}.pt"

try:
    generator_pt = Generator(noise_dim=NOISE_DIM, num_classes=NUM_CLASSES)
    state = torch.load(pt_ckpt_path, map_location="cpu")
    generator_pt.load_state_dict(state)
    generator_pt.eval()
    print(f"Loaded PyTorch generator (epoch {PYTORCH_EPOCH}) from: {pt_ckpt_path}")
    PT_AVAILABLE = True
except FileNotFoundError:
    print(f"[WARNING] Checkpoint not found: {pt_ckpt_path}")
    print("Copy generator_*.pt files from CIT into weights/pytorch/ (see README there).")
    PT_AVAILABLE = False
except Exception as e:
    print(f"[WARNING] Could not load PyTorch generator: {e}")
    PT_AVAILABLE = False

In [ ]:
if PT_AVAILABLE:
    num_examples = NUM_CLASSES * SAMPLES_PER_CLASS
    class_vectors_pt = np.repeat(np.eye(NUM_CLASSES), SAMPLES_PER_CLASS, axis=0)

    with torch.no_grad():
        noise_pt   = torch.randn(num_examples, NOISE_DIM)
        class_t    = torch.tensor(class_vectors_pt, dtype=torch.float32)
        X_fake_pt  = generator_pt(noise_pt, class_t).numpy()

    y_fake_pt = np.repeat(LABEL_ORDER, SAMPLES_PER_CLASS)
    print(f"PyTorch generated : {X_fake_pt.shape}  ({SAMPLES_PER_CLASS} per class)")
else:
    X_fake_pt = None
    y_fake_pt = None

In [ ]:
if PT_AVAILABLE:
    plot_per_class_row(X_fake_pt, y_fake_pt, LABEL_ORDER,
                       color="C2", save_name="pt_generated_waveforms_row")

### 3a. PyTorch — checkpoint comparison

One vertex sample per class for each downloaded checkpoint. Quick visual check of how output quality evolves across training.

In [ ]:
PT_COMPARE_EPOCHS = [150, 200, 250, 500]   # adjust to whichever .pt files you have

fig, axes = plt.subplots(
    len(PT_COMPARE_EPOCHS), NUM_CLASSES,
    figsize=(2.8 * NUM_CLASSES, 2.2 * len(PT_COMPARE_EPOCHS))
)

with torch.no_grad():
    noise_v = torch.randn(NUM_CLASSES, NOISE_DIM)   # fixed noise so shapes are comparable
    class_vecs = torch.eye(NUM_CLASSES, dtype=torch.float32)

    for row, epoch in enumerate(PT_COMPARE_EPOCHS):
        ckpt = PYTORCH_WEIGHTS_DIR / f"generator_{epoch}.pt"
        if not ckpt.exists():
            print(f"[SKIP] generator_{epoch}.pt not found in weights/pytorch/")
            for ax in axes[row]: ax.axis("off")
            continue

        gen = Generator(noise_dim=NOISE_DIM, num_classes=NUM_CLASSES)
        gen.load_state_dict(torch.load(ckpt, map_location="cpu"))
        gen.eval()
        signals = gen(noise_v, class_vecs).numpy()

        for col, (sig, lbl) in enumerate(zip(signals, LABEL_ORDER)):
            ax = axes[row, col]
            ax.plot(sig, lw=0.7, color=f"C{row + 1}")
            ax.set_xticks([]); ax.set_yticks([])
            if row == 0:
                ax.set_title(lbl.replace("_", " "), fontsize=11)
            if col == 0:
                ax.set_ylabel(f"epoch {epoch}", fontsize=10,
                              rotation=0, labelpad=48, va="center")

plt.suptitle("PyTorch cDVGAN — vertex samples across checkpoints", fontsize=13)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "pytorch_checkpoint_comparison.pdf", bbox_inches="tight")
plt.show()

### 3b. TF (epoch 210) vs PyTorch — direct comparison

Set `COMPARE_PT_EPOCH` to whichever checkpoint you want to compare against the TF model.
Downloaded options: 150, 200, 250, 500. Grab 225 separately if you want the exact midpoint.

In [ ]:
COMPARE_PT_EPOCH = 250   # ← change to 200, 225, etc.

ckpt_compare = PYTORCH_WEIGHTS_DIR / f"generator_{COMPARE_PT_EPOCH}.pt"
gen_compare = Generator(noise_dim=NOISE_DIM, num_classes=NUM_CLASSES)
gen_compare.load_state_dict(torch.load(ckpt_compare, map_location="cpu"))
gen_compare.eval()

with torch.no_grad():
    noise_v  = torch.randn(NUM_CLASSES, NOISE_DIM)
    class_vecs = torch.eye(NUM_CLASSES, dtype=torch.float32)
    signals_pt_compare = gen_compare(noise_v, class_vecs).numpy()

# Generate one TF vertex sample per class
if TF_AVAILABLE:
    import tensorflow as tf
    noise_tf_v   = tf.random.normal(shape=(NUM_CLASSES, NOISE_DIM))
    signals_tf_v = tf_generator.predict([noise_tf_v, np.eye(NUM_CLASSES)], verbose=0)
else:
    signals_tf_v = None

# ── Plot: TF row on top, PyTorch row below ────────────────────────────────────
fig, axes = plt.subplots(2, NUM_CLASSES, figsize=(2.8 * NUM_CLASSES, 4.5))

rows = [
    (signals_tf_v,        "TF  ep.210", "C1"),
    (signals_pt_compare,  f"PT  ep.{COMPARE_PT_EPOCH}", "C2"),
]
for row_idx, (signals, label, color) in enumerate(rows):
    if signals is None:
        for ax in axes[row_idx]: ax.axis("off")
        continue
    for col, (sig, lbl) in enumerate(zip(signals, LABEL_ORDER)):
        ax = axes[row_idx, col]
        ax.plot(sig, lw=0.8, color=color)
        ax.set_xticks([]); ax.set_yticks([])
        if row_idx == 0:
            ax.set_title(lbl.replace("_", " "), fontsize=11)
        if col == 0:
            ax.set_ylabel(label, fontsize=10, rotation=0, labelpad=52, va="center")

plt.suptitle(f"TF epoch 210 vs PyTorch epoch {COMPARE_PT_EPOCH} — vertex samples", fontsize=13)
plt.tight_layout()
plt.savefig(PLOTS_DIR / f"tf_vs_pt_ep{COMPARE_PT_EPOCH}.pdf", bbox_inches="tight")
plt.show()

## 4. Waveform Comparison

Side-by-side grid: rows = glitch classes, columns = Real / TF / PyTorch.

In [ ]:
def plot_comparison_grid(sources, labels, n_examples=5, save_name=None):
    """
    sources : list of (X, y, title, color)
    labels  : class label order
    """
    n_sources = len(sources)
    n_classes = len(labels)
    fig, axes = plt.subplots(
        n_classes, n_sources * n_examples,
        figsize=(2.5 * n_sources * n_examples, 1.8 * n_classes)
    )

    for col_group, (X, y, title, color) in enumerate(sources):
        if X is None:
            continue
        for row, lbl in enumerate(labels):
            idx = np.where(y == lbl)[0]
            chosen = np.random.choice(idx, min(n_examples, len(idx)), replace=False)
            for j, ci in enumerate(chosen):
                col = col_group * n_examples + j
                ax = axes[row, col]
                ax.plot(X[ci], lw=0.6, color=color)
                ax.set_xticks([])
                ax.set_yticks([])
                if row == 0:
                    ax.set_title(f"{title}\n{j+1}" if j == 0 else str(j + 1), fontsize=9)
                if col == 0:
                    ax.set_ylabel(lbl.replace("_", "\n"), fontsize=8,
                                  rotation=0, labelpad=45, va="center")

    plt.suptitle("Glitch Waveforms: Real vs TF vs PyTorch", fontsize=13, y=1.01)
    plt.tight_layout()
    if save_name:
        plt.savefig(PLOTS_DIR / f"{save_name}.pdf", bbox_inches="tight")
    plt.show()


sources = [
    (X_real,    y_real,    "Real",    "C0"),
    (X_fake_tf, y_fake_tf, "TF",      "C1"),
    (X_fake_pt, y_fake_pt, "PyTorch", "C2"),
]
plot_comparison_grid(sources, LABEL_ORDER, n_examples=3, save_name="waveform_comparison_grid")

## 5. UMAP Dimensionality Reduction

Joint 3D UMAP embedding of real + TF-generated + PyTorch-generated signals.
Uses a random subsample per class to keep the computation tractable.

In [ ]:
# ── Subsample for UMAP ────────────────────────────────────────────────────────
N_UMAP = 300   # samples per class per source

def subsample(X, y, label_order, n):
    idx = []
    for lbl in label_order:
        cls_idx = np.where(y == lbl)[0]
        idx.extend(np.random.choice(cls_idx, min(n, len(cls_idx)), replace=False))
    return X[idx], y[idx]

chunks_X, chunks_y, chunks_domain = [], [], []

X_r_sub, y_r_sub = subsample(X_real, y_real, LABEL_ORDER, N_UMAP)
chunks_X.append(X_r_sub); chunks_y.append(y_r_sub)
chunks_domain.extend(["Real"] * len(X_r_sub))

if TF_AVAILABLE:
    X_tf_sub, y_tf_sub = subsample(X_fake_tf, y_fake_tf, LABEL_ORDER, N_UMAP)
    chunks_X.append(X_tf_sub); chunks_y.append(y_tf_sub)
    chunks_domain.extend(["TF"] * len(X_tf_sub))

if PT_AVAILABLE:
    X_pt_sub, y_pt_sub = subsample(X_fake_pt, y_fake_pt, LABEL_ORDER, N_UMAP)
    chunks_X.append(X_pt_sub); chunks_y.append(y_pt_sub)
    chunks_domain.extend(["PyTorch"] * len(X_pt_sub))

X_all = np.concatenate(chunks_X, axis=0)
y_all = np.concatenate(chunks_y, axis=0)
domain_all = np.array(chunks_domain)

print(f"UMAP input: {X_all.shape}  (total samples)")

In [ ]:
# ── Fit UMAP ─────────────────────────────────────────────────────────────────
reducer = umap.UMAP(
    n_components=3,
    n_neighbors=30,
    min_dist=0.3,
    metric="euclidean",
    random_state=42,
)
embedding = reducer.fit_transform(X_all)
print(f"Embedding shape: {embedding.shape}")

In [ ]:
# ── Remove outliers (IQR-based on per-axis 1st/99th percentile) ───────────────
def remove_outliers(emb, pct=1):
    lo = np.percentile(emb, pct, axis=0)
    hi = np.percentile(emb, 100 - pct, axis=0)
    mask = np.all((emb >= lo) & (emb <= hi), axis=1)
    return mask

mask = remove_outliers(embedding, pct=1)
emb_clean     = embedding[mask]
y_clean       = y_all[mask]
domain_clean  = domain_all[mask]
print(f"After outlier removal: {emb_clean.shape[0]} / {embedding.shape[0]} points")

In [ ]:
# ── 3D UMAP plot ──────────────────────────────────────────────────────────────
from matplotlib.lines import Line2D

CLASS_COLORS = dict(zip(LABEL_ORDER, [f"C{i}" for i in range(len(LABEL_ORDER))]))
DOMAIN_MARKERS = {"Real": "o", "TF": "^", "PyTorch": "s"}
DOMAIN_SIZES   = {"Real": 6,  "TF": 8,   "PyTorch": 8}
DOMAIN_ALPHA   = {"Real": 0.4,"TF": 0.6, "PyTorch": 0.6}

fig = plt.figure(figsize=(12, 9))
ax  = fig.add_subplot(111, projection="3d")

for domain in ["Real", "TF", "PyTorch"]:
    if domain not in domain_clean:
        continue
    for lbl in LABEL_ORDER:
        mask_d = (domain_clean == domain) & (y_clean == lbl)
        if mask_d.sum() == 0:
            continue
        ax.scatter(
            emb_clean[mask_d, 0], emb_clean[mask_d, 1], emb_clean[mask_d, 2],
            c=CLASS_COLORS[lbl],
            marker=DOMAIN_MARKERS[domain],
            s=DOMAIN_SIZES[domain],
            alpha=DOMAIN_ALPHA[domain],
            label=f"{lbl} ({domain})",
        )

# Build compact legend: one entry per class (color) + one per domain (marker)
class_handles = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor=CLASS_COLORS[lbl],
           markersize=8, label=lbl.replace("_", " "))
    for lbl in LABEL_ORDER
]
domain_handles = [
    Line2D([0], [0], marker=DOMAIN_MARKERS[d], color="grey",
           markersize=8, label=d)
    for d in DOMAIN_MARKERS if d in domain_clean
]
ax.legend(handles=class_handles + domain_handles,
          loc="upper left", fontsize=7, ncol=2, framealpha=0.6)

ax.set_xlabel("UMAP 1"); ax.set_ylabel("UMAP 2"); ax.set_zlabel("UMAP 3")
ax.set_title("3D UMAP: Real vs TF vs PyTorch")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "umap_3d_comparison.pdf", bbox_inches="tight")
plt.show()

## 6. Gravity Spy Classification

Classifies the generated signals using the Gravity Spy CNN.

**Requirements:** `gengli`, `gwpy`, `gravityspy` and access to an online GW frame channel.
Adjust the configuration block below before running.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
IFO            = "H1"
SRATE          = 4096
GW_START       = 1262540000       # GPS start for background noise segment
GW_END         = GW_START + 40
CHANNEL        = f"{IFO}:GDS-CALIB_STRAIN"
PATH_TO_MODEL  = PROJECT_ROOT / "models" / "sidd-cqg-paper-O3-model.h5"
PATH_TO_REPO   = "/home/meesde.boer/gw_learn/GravitySpy/"  # path on CIT (only needed for imports)
NUM_CLASSIFY   = 100              # signals to classify per class
SNR_TARGET     = 150

In [ ]:
import sys, os
sys.path.insert(0, PATH_TO_REPO)

import numpy as np
from tqdm import tqdm
from gwpy.timeseries import TimeSeries
from gravityspy.classify import classify
import pandas as pd
from cdvgan.utils import whitened_snr_scaling

def classify_generated_signals(X_gen, y_gen, label_order, tag):
    """Inject generated signals into background noise and classify with Gravity Spy."""
    # Fetch background noise
    strain_bg = TimeSeries.fetch(CHANNEL, GW_START, GW_END)
    strain_bg = strain_bg.resample(SRATE)

    rows = []
    for lbl in tqdm(label_order, desc=f"Classifying [{tag}]"):
        cls_idx = np.where(y_gen == lbl)[0]
        chosen  = np.random.choice(cls_idx, min(NUM_CLASSIFY, len(cls_idx)), replace=False)
        for i in chosen:
            signal = X_gen[i]
            # Scale signal to target SNR and inject into noise segment
            # (whitened_snr_scaling from the original utils.py)
            # Simplified injection: normalise to unit RMS then scale
            signal = signal / (signal.std() + 1e-8) * SNR_TARGET
            t0     = GW_START + 20
            ts     = TimeSeries(signal, t0=t0, dt=1.0 / SRATE, name=CHANNEL)
            injected = strain_bg.inject(ts)
            result = classify(injected, PATH_TO_MODEL, ifo=IFO)
            rows.append({"true_label": lbl, "pred_label": result["label"],
                         "confidence": result["confidence"], "tag": tag})
    return pd.DataFrame(rows)

# Run classification for whichever generators are available
df_tf = classify_generated_signals(X_fake_tf, y_fake_tf, LABEL_ORDER, "TF") \
        if TF_AVAILABLE else None
df_pt = classify_generated_signals(X_fake_pt, y_fake_pt, LABEL_ORDER, "PyTorch") \
        if PT_AVAILABLE else None

if df_tf is not None: df_tf.to_csv(PLOTS_DIR / "gspy_results_tf.csv", index=False)
if df_pt is not None: df_pt.to_csv(PLOTS_DIR / "gspy_results_pt.csv", index=False)

In [ ]:
# ── Confusion matrices ────────────────────────────────────────────────────────
import seaborn as sns

def plot_confusion(df, tag, save_name):
    if df is None:
        print(f"No results for {tag}")
        return None

    count_matrix = pd.crosstab(
        df["true_label"], df["pred_label"],
        rownames=["True"], colnames=["Predicted"]
    ).reindex(index=LABEL_ORDER, columns=LABEL_ORDER, fill_value=0)

    acc = np.trace(count_matrix.values) / count_matrix.values.sum()

    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(count_matrix, annot=True, fmt="d", cmap="Blues",
                xticklabels=[l.replace("_", "\n") for l in LABEL_ORDER],
                yticklabels=[l.replace("_", "\n") for l in LABEL_ORDER],
                ax=ax)
    ax.set_title(f"Gravity Spy — {tag} generated   (accuracy = {acc:.1%})")
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / f"{save_name}.pdf", bbox_inches="tight")
    plt.show()
    print(f"{tag} accuracy: {acc:.3f}")
    return acc

acc_tf = plot_confusion(df_tf, "TF (epoch 210)", "gspy_confusion_tf")
acc_pt = plot_confusion(df_pt, f"PyTorch (epoch {PYTORCH_EPOCH})", "gspy_confusion_pt")

### Load pre-saved classification results

If you have already run the classifier and saved CSV files, use this cell instead.

In [ ]:
# Uncomment and adjust paths as needed
# df_tf = pd.read_csv(PLOTS_DIR / "gspy_results_tf.csv")
# df_pt = pd.read_csv(PLOTS_DIR / "gspy_results_pt.csv")
# acc_tf = plot_confusion(df_tf, "TF (epoch 210)", "gspy_confusion_tf_loaded")
# acc_pt = plot_confusion(df_pt, f"PyTorch (epoch {PYTORCH_EPOCH})", "gspy_confusion_pt_loaded")

## 7. Summary

In [ ]:
import pandas as pd

summary_rows = [{"Model": "Real data (reference)", "Epoch": "—",
                 "GravitySpy accuracy": "—"}]

if TF_AVAILABLE:
    summary_rows.append({
        "Model": "TF cDVGAN", "Epoch": 210,
        "GravitySpy accuracy": f"{acc_tf:.3f}" if acc_tf is not None else "not run"
    })

if PT_AVAILABLE:
    summary_rows.append({
        "Model": "PyTorch cDVGAN", "Epoch": PYTORCH_EPOCH,
        "GravitySpy accuracy": f"{acc_pt:.3f}" if acc_pt is not None else "not run"
    })

summary = pd.DataFrame(summary_rows).set_index("Model")
print(summary.to_string())
summary.to_csv(PLOTS_DIR / "summary.csv")